# Bulls and cows - can we solve it using entropy

In [5]:
import pickle
import random
import os
from collections.abc import Callable
from dataclasses import dataclass
from enum import Enum
from typing import Protocol
from collections import Counter
import math

import numpy as np


In [6]:
class GameState(Enum):
    IN_PROGRESS = "in_progress"
    WON = "won"
    LOST = "lost"


class BullsAndCowsGame:
    def __init__(self, external_secret = None):
        self.game_state = GameState.IN_PROGRESS
        self.guess_count = 0

        self.secret = external_secret or str(random.randint(1000, 9999))


    def make_guess(self, guess: str) -> tuple[int, int]:
        if self.game_state != GameState.IN_PROGRESS:
            raise RuntimeError("The game is not in progress")

        if len(guess) != len(self.secret):
            raise RuntimeError(f"Incorrect format: {guess}")

        self.guess_count += 1

        bulls, cows = BullsAndCowsGame.detect_bulls_and_cows(self.secret, guess)

        if bulls == 4:
            self.game_state = GameState.WON

        return (bulls, cows)


    @staticmethod
    def detect_bulls_and_cows(secret: str, guess: str) -> tuple[int, int]:
        bulls = 0
        secret_counts = [0] * 10
        guess_counts = [0] * 10

        for s, g in zip(secret, guess):
            if s == g:
                bulls += 1
                continue

            secret_counts[ord(s) - ord('0')] += 1
            guess_counts[ord(g) - ord('0')] += 1

        cows = sum(
            min(s, g)
            for s, g in zip(secret_counts, guess_counts)
        )

        return bulls, cows


In [7]:

test_secret = "1233"
game = BullsAndCowsGame(external_secret = test_secret)

def assert_guess(actual, expected):
    assert actual == expected, f"expected {expected}, got {actual}"


assert_guess(game.make_guess("1332"), (2, 2))
assert_guess(game.make_guess("5111"), (0, 1))
assert_guess(game.make_guess("1331"), (2, 1))
assert_guess(game.make_guess("3339"), (1, 1))
assert_guess(game.make_guess("3333"), (2, 0))
assert_guess(game.make_guess("5678"), (0, 0))
assert_guess(game.make_guess("1256"), (2, 0))
assert_guess(game.make_guess("3321"), (0, 4))
assert_guess(game.make_guess("3312"), (0, 4))
assert_guess(game.make_guess("3789"), (0, 1))
assert_guess(game.make_guess("3389"), (0, 2))
assert_guess(game.make_guess("8911"), (0, 1))
assert_guess(BullsAndCowsGame.detect_bulls_and_cows("1122", "2211"), (0, 4))
assert_guess(game.make_guess(test_secret), (4, 0))
assert_guess(game.game_state, GameState.WON)

In [8]:
play_game = False

if play_game:
    game = BullsAndCowsGame()

    while(True):
        guess = input("What is your guess?")
        if guess == 'q':
            print(f"The number was: {game.secret}")
            break
        bulls, cows = game.make_guess(guess)
        if game.game_state == GameState.WON:
            print(f"You won with: {guess} in {game.guess_count}")
            break
        else:
            print(f"Turn: {game.guess_count} - guess: {guess} - bulls: {bulls}, cows: {cows}")



# Play the game

In [9]:

class PlayStrategy(Protocol):

    def generate_guess(self) -> str:
        ...

    def record_turn_feedback(self, bulls:int, cows: int):
        ...

In [10]:
class IterateThroughAllPossibilitiesStrategy(PlayStrategy):
    def __init__(self):
       self.guess_tracker = 999

    def generate_guess(self) -> str:
        self.guess_tracker += 1

        if self.guess_tracker >= 10000:
            raise RuntimeError("The strategy has exhausted all possible guesses")

        return str(self.guess_tracker)

    def record_turn_feedback(self, bulls:int, cows: int):
        pass


In [41]:
@dataclass(frozen=True, slots=True)
class GameResult:
    game_state: GameState
    number_of_turns: int


@dataclass(frozen=True, slots=True)
class FacilitatorRunSummary:
    total_games: int
    won_games: int
    average_turns: float
    all_games_results: list[GameResult]

class GameFacilitator:
    def __init__(
            self,
            game_factory: Callable[[], BullsAndCowsGame],
            strategy_factory: Callable[[], PlayStrategy]):

        self.game_factory = game_factory
        self.strategy_factory = strategy_factory


    def play_games(self, count: int, turn_frequency_log: int = 100) -> FacilitatorRunSummary:
        game_results: list[GameResult] = []

        for i in range(count):
            game = self.game_factory()
            strategy = self.strategy_factory()

            if i % turn_frequency_log == 0:
                print("Game: ", i)

            turn_counter = 0
            while True:
                turn_counter += 1

                guess = strategy.generate_guess()
                bulls, cows = game.make_guess(guess)
                strategy.record_turn_feedback(bulls, cows)

                if game.game_state != GameState.IN_PROGRESS:
                    result = GameResult(game.game_state, turn_counter)
                    game_results.append(result)
                    break

        run_result = FacilitatorRunSummary(
            total_games=count,
            won_games = sum(1 for x in game_results if x.game_state == GameState.WON),
            average_turns = sum(x.number_of_turns for x in game_results) / len(game_results),
            all_games_results = game_results
        )

        return run_result




In [12]:
game_factory = lambda: BullsAndCowsGame()
strategy_factory = lambda: IterateThroughAllPossibilitiesStrategy()

facilitator = GameFacilitator(game_factory, strategy_factory)

facilitator.play_games(10)


Game:  0


FacilitatorRunSummary(total_games=10, won_games=10, average_turns=4943.9, all_games_results=[GameResult(game_state=<GameState.WON: 'won'>, number_of_turns=8532), GameResult(game_state=<GameState.WON: 'won'>, number_of_turns=8580), GameResult(game_state=<GameState.WON: 'won'>, number_of_turns=2705), GameResult(game_state=<GameState.WON: 'won'>, number_of_turns=3578), GameResult(game_state=<GameState.WON: 'won'>, number_of_turns=1451), GameResult(game_state=<GameState.WON: 'won'>, number_of_turns=3103), GameResult(game_state=<GameState.WON: 'won'>, number_of_turns=5404), GameResult(game_state=<GameState.WON: 'won'>, number_of_turns=5111), GameResult(game_state=<GameState.WON: 'won'>, number_of_turns=8979), GameResult(game_state=<GameState.WON: 'won'>, number_of_turns=1996)])

# Solve it choosing next turn based on information gain

In [13]:
sum(1 for x in range(1000, 10000))

9000

In [14]:
# I would like to solve the game of Bulls And Cows using information gain analysis
# information that an observation gives: i = -log(p), where p is the probability of the observation
# entropy: e[i] = sum(-p*log(p)) - or expected value of the information, this is a measurement of how much uncertainty there is in the entire set

# So what is the initial entropy/uncertainty for Bulls and Cows?
possibilities = 9000
p = 1 / possibilities

pos_array = np.full(possibilities, p)

entropy_initial = sum(-p * np.log2(pos_array))
print("initial entropy:", entropy_initial)
# initial entropy: 13.135709286102902
# What does that mean? That you should be able to encode every initial state of the game in 13.13 binary questions


initial entropy: 13.135709286102902


In [15]:
# Algo:
# OUTER LOOP: for each possibility [1000 to 9999],
# counter = Counter
# INNER LOOP: for each other possibility [1000 to 9999]
#   - find the number of bulls and cows.
#   - counter[(bulls, cows)]
# compute -sum( (count/total) * log2(count/total) )
# Example:
    # outer: X [1000, 9999]
    # inner: 1000...Y....9999
    # (bulls, cows) = detect_bulls_cows(inner, outer)
    # counter[(bulls, cows)] += 1
    # E[I] for Y = sum(-p*)*log2(p)


def calc_entropy(guess: str, possibilities: list[str]) -> 'float':
    counter = Counter()

    for secret in possibilities:
        bulls_and_cows = BullsAndCowsGame.detect_bulls_and_cows(str(secret), str(guess))
        counter[bulls_and_cows] += 1

    total = counter.total()
    entropy = sum(-(count / total) * np.log2(count / total) for count in counter.values())

    return entropy


possibilities = range(1000, 10000)
sorted([
    (calc_entropy(str(guess), possibilities), guess)
    for guess in [1000, 1111, 1234, 5678, 1122, 9999, 1223, 1222]
], reverse=True)

[(np.float64(2.7494100494795033), 5678),
 (np.float64(2.749410049479503), 1234),
 (np.float64(2.6014411615807576), 1223),
 (np.float64(2.3190728663039275), 1122),
 (np.float64(2.197558084624638), 1222),
 (np.float64(1.9447337794096033), 1000),
 (np.float64(1.177347499747682), 1111),
 (np.float64(1.1773474997476818), 9999)]

In [16]:
if os.path.exists("entropy_9k.pkl"):
    with open("entropy_9k.pkl", "rb") as file:
        entropy_9k = pickle.load(file)
else:
    entropy_9k = {}
    possibilities = range(1000, 10000)
    for guess in possibilities:
        if guess % 1000 == 0:
            print(guess)

        entropy = calc_entropy(str(guess), possibilities)
        entropy_9k[guess] = entropy

    # Save
    with open("entropy_9k.pkl", "wb") as file:
        pickle.dump(entropy_9k, file)


In [17]:
# cache

if os.path.exists("bulls_cows_9k.pkl"):
    with open("bulls_cows_9k.pkl", "rb") as file:
        bulls_cows_cache = pickle.load(file)
else:
    possibilities = range(1000, 10000)

    bulls_cows_cache = {}

    for guess in possibilities:
        if guess % 1000 == 0:
            print(guess)

        bulls_cows_cache[guess] = {}

        for secret in possibilities:
            bulls, cows = BullsAndCowsGame.detect_bulls_and_cows(
                str(secret),
                str(guess)
            )

            # Encode (bulls, cows) as a single integer
            bulls_cows_cache[guess][secret] = bulls * 10 + cows

    with open("bulls_cows_9k.pkl", "wb") as file:
        pickle.dump(bulls_cows_cache, file)

In [18]:
def calc_entropy_optimized(guess: str, possibilities: list[str]) -> 'float':
    counter = Counter()

    for secret in possibilities:
        bulls_and_cows = bulls_cows_cache[int(guess)][int(secret)]
        counter[bulls_and_cows] += 1

    total = counter.total()
    entropy = sum(-(count / total) * np.log2(count / total) for count in counter.values())

    return entropy

In [19]:
possibilities = range(1000, 10000)
sorted([
    (calc_entropy_optimized(str(guess), possibilities), guess)
    for guess in [1000, 1111, 1234, 5678, 1122, 9999, 1223, 1222]
], reverse=True)

[(np.float64(2.7494100494795033), 5678),
 (np.float64(2.749410049479503), 1234),
 (np.float64(2.6014411615807576), 1223),
 (np.float64(2.3190728663039275), 1122),
 (np.float64(2.197558084624638), 1222),
 (np.float64(1.9447337794096033), 1000),
 (np.float64(1.177347499747682), 1111),
 (np.float64(1.1773474997476818), 9999)]

In [20]:
candidates = sorted( [(key, val) for key, val in entropy_9k.items()], key=lambda x: x[1], reverse=True )
max_val = candidates[0][1]
candidates = [str(candidate) for candidate, entropy in candidates if entropy == max_val]
guess = random.choice(candidates)
guess


'7635'

In [21]:
reduced_set = [
    candidate
    for candidate in range(1000, 10000)
    if BullsAndCowsGame.detect_bulls_and_cows(str(candidate), guess) == (2, 2)
]
print(reduced_set)

[3675, 5637, 6735, 7365, 7536, 7653]


In [47]:
class EntropyStrategy(PlayStrategy):
    def __init__(self, first_turn_candidates):
        self.turns_counter = 0

        self._first_turn_candidates = first_turn_candidates

        self._last_guess = None
        self._feedbacks = []
        self._remaining_possibilities = [str(x) for x in range(1000, 10000)]


    def generate_guess(self) -> str:
        self.turns_counter += 1
        if self.turns_counter == 1:
            guess = random.choice(candidates)
        else:
            last_feedback = self._feedbacks[-1]

            # print(f"old set: {len(self._remaining_possibilities)}")
            new_set = self.reduce_set(self._remaining_possibilities, self._last_guess, last_feedback)
            # print(f"new set: {len(new_set)}")

            self._remaining_possibilities = new_set

            max_entropy = -1
            guess = None
            for candidate in self._remaining_possibilities:
                entropy = calc_entropy_optimized(candidate, possibilities = self._remaining_possibilities)

                if entropy >= max_entropy:
                    guess = candidate
                    max_entropy = entropy


        self._last_guess = guess

        return guess


    def record_turn_feedback(self, bulls:int, cows: int):
        self._feedbacks.append( (bulls, cows) )


    @staticmethod
    def reduce_set(possibilities: list[str], guess: str, bulls_and_cows: tuple[int, int]) -> list[str]:
        reduced_set = [
            candidate for candidate in possibilities
            if BullsAndCowsGame.detect_bulls_and_cows(str(candidate), guess) == bulls_and_cows
        ]

        return reduced_set


In [49]:
facilitator_entropy = GameFacilitator(
    lambda: BullsAndCowsGame(),
    lambda: EntropyStrategy(first_turn_candidates=candidates))

result = facilitator_entropy.play_games(100)
print(result)
print("All: ", result.total_games)
print("Won: ", result.won_games)
print("Average turns: ", result.average_turns)
print("Max: ", max(x.number_of_turns for x in result.all_games_results) )
print("Min: ", min(x.number_of_turns for x in result.all_games_results) )

# All:  1000
# Won:  1000
# Average turns:  5.728
# Max:  9
# Min:  3


Game:  0
FacilitatorRunSummary(total_games=100, won_games=100, average_turns=5.64, all_games_results=[GameResult(game_state=<GameState.WON: 'won'>, number_of_turns=6), GameResult(game_state=<GameState.WON: 'won'>, number_of_turns=6), GameResult(game_state=<GameState.WON: 'won'>, number_of_turns=5), GameResult(game_state=<GameState.WON: 'won'>, number_of_turns=5), GameResult(game_state=<GameState.WON: 'won'>, number_of_turns=6), GameResult(game_state=<GameState.WON: 'won'>, number_of_turns=7), GameResult(game_state=<GameState.WON: 'won'>, number_of_turns=6), GameResult(game_state=<GameState.WON: 'won'>, number_of_turns=5), GameResult(game_state=<GameState.WON: 'won'>, number_of_turns=4), GameResult(game_state=<GameState.WON: 'won'>, number_of_turns=5), GameResult(game_state=<GameState.WON: 'won'>, number_of_turns=6), GameResult(game_state=<GameState.WON: 'won'>, number_of_turns=6), GameResult(game_state=<GameState.WON: 'won'>, number_of_turns=4), GameResult(game_state=<GameState.WON: 'w

In [60]:
import plotly.express as px

turns = [x.number_of_turns for x in result.all_games_results]

fig = px.histogram(
    x=turns,
    nbins=max(turns) - min(turns) + 1,
    range_x=[min(turns), max(turns)],
    title="Distribution of turns per game",
    labels={"x": "Number of turns", "y": "Games"},
)

fig.update_layout(
    bargap=0.1,
    xaxis=dict(dtick=1),
    template="plotly_white"
)

fig.show()

[6,
 6,
 5,
 5,
 6,
 7,
 6,
 5,
 4,
 5,
 6,
 6,
 4,
 2,
 6,
 7,
 5,
 5,
 7,
 6,
 4,
 7,
 6,
 6,
 6,
 6,
 7,
 6,
 6,
 5,
 6,
 5,
 6,
 6,
 8,
 4,
 6,
 5,
 4,
 6,
 6,
 6,
 7,
 7,
 4,
 8,
 5,
 7,
 6,
 6,
 3,
 7,
 6,
 5,
 5,
 6,
 4,
 6,
 6,
 6,
 7,
 5,
 4,
 6,
 5,
 4,
 6,
 4,
 9,
 5,
 5,
 6,
 6,
 6,
 7,
 5,
 4,
 6,
 7,
 6,
 5,
 5,
 7,
 6,
 5,
 5,
 6,
 6,
 7,
 4,
 3,
 6,
 4,
 6,
 6,
 6,
 6,
 6,
 6,
 6]

In [ ]:
# Using ALL possibilities for a guess, not just those that are still possible
# All:  100
# Won:  100
# Average turns: e

# Max:  10
# Min:  2

In [52]:
Min: 3

All:  1000
Won:  1000
Average turns:  6.907
Max:  11
Min:  3


In [55]:
# my_secret = random.randint(1000, 10000)
# my_secret = str(my_secret)
my_secret = "9999"
turns_counter = 0

solver = EntropyStrategy(first_turn_candidates=candidates)
print("Secret is: ", my_secret)
while True:
    turns_counter += 1
    guess = solver.generate_guess()
    bulls, cows = BullsAndCowsGame.detect_bulls_and_cows(my_secret, guess)
    print(f"guess: {guess}. B&C: {(bulls, cows)}")
    solver.record_turn_feedback( bulls, cows )

    if bulls == 4:
        print(f"game won in: {turns_counter} turns")
        break

Secret is:  9999
guess: 3826. B&C: (0, 0)
guess: 9771. B&C: (1, 0)
guess: 9509. B&C: (2, 0)
guess: 9449. B&C: (2, 0)
guess: 9545. B&C: (1, 0)
guess: 9999. B&C: (4, 0)
game won in: 6 turns


In [33]:
class BaselineStrategy(PlayStrategy):
    def __init__(self):
        self.turns_counter = 0

        self._last_guess = None
        self._feedbacks = []
        self._remaining_possibilities = [str(x) for x in range(1000, 10000)]


    def generate_guess(self) -> str:
        self.turns_counter += 1
        if self.turns_counter == 1:
            guess = random.choice(["1234", "2345", "3456", "4567", "5678", "6789"])
        else:
            last_feedback = self._feedbacks[-1]

            new_set = self.reduce_set(self._remaining_possibilities, self._last_guess, last_feedback)
            self._remaining_possibilities = new_set

            guess = random.choice(self._remaining_possibilities)

        self._last_guess = guess

        return guess


    def record_turn_feedback(self, bulls:int, cows: int):
        self._feedbacks.append( (bulls, cows) )


    @staticmethod
    def reduce_set(possibilities: list[str], guess: str, bulls_and_cows: tuple[int, int]) -> list[str]:
        reduced_set = [
            candidate for candidate in possibilities
            if BullsAndCowsGame.detect_bulls_and_cows(str(candidate), guess) == bulls_and_cows
        ]

        return reduced_set


In [37]:
facilitator_baseline = GameFacilitator(
    lambda: BullsAndCowsGame(),
    lambda: BaselineStrategy())

result = facilitator_entropy.play_games(1000)
print(result)
print("All: ", result.total_games)
print("Won: ", result.won_games)
print("Average turns: ", result.average_turns)
print("Max: ", max(x.number_of_turns for x in result.all_games_results) )
print("Min: ", min(x.number_of_turns for x in result.all_games_results) )

Game:  0
Game:  100
Game:  200
Game:  300
Game:  400
Game:  500
Game:  600
Game:  700
Game:  800
Game:  900
FacilitatorRunSummary(total_games=1000, won_games=1000, average_turns=6.115, all_games_results=[GameResult(game_state=<GameState.WON: 'won'>, number_of_turns=7), GameResult(game_state=<GameState.WON: 'won'>, number_of_turns=5), GameResult(game_state=<GameState.WON: 'won'>, number_of_turns=9), GameResult(game_state=<GameState.WON: 'won'>, number_of_turns=5), GameResult(game_state=<GameState.WON: 'won'>, number_of_turns=6), GameResult(game_state=<GameState.WON: 'won'>, number_of_turns=7), GameResult(game_state=<GameState.WON: 'won'>, number_of_turns=5), GameResult(game_state=<GameState.WON: 'won'>, number_of_turns=5), GameResult(game_state=<GameState.WON: 'won'>, number_of_turns=5), GameResult(game_state=<GameState.WON: 'won'>, number_of_turns=6), GameResult(game_state=<GameState.WON: 'won'>, number_of_turns=6), GameResult(game_state=<GameState.WON: 'won'>, number_of_turns=6), Game